In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e4/train.csv
/kaggle/input/competitions/playground-series-s6e4/test.csv


In [2]:
data= pd.read_csv("/kaggle/input/competitions/playground-series-s6e4/train.csv")

In [3]:
def data_info(data):
    print(data.head())
    print('-' * 100)
    print(data.describe())
    print('-' * 100)
    print(data.info())
    print('-' * 100)
    print(f"Number of Nulls: {data.isna().sum().sum()}")
    print('-' * 100)
    print(f"Number of duplicates: {data.duplicated().sum()}")
    print('-' * 100)
    print(f"Rows/Samples: {data.shape[0]}\nColumns/Features: {data.shape[1]}")
    

data_info(data)

   id Soil_Type  Soil_pH  Soil_Moisture  Organic_Carbon  \
0   0     Loamy     4.92          32.58            1.01   
1   1      Clay     7.08          56.61            0.44   
2   2      Clay     5.69          27.71            0.81   
3   3     Sandy     5.65          13.32            1.33   
4   4      Clay     7.96          59.14            0.38   

   Electrical_Conductivity  Temperature_C  Humidity  Rainfall_mm  \
0                     3.05          15.01     50.61       725.99   
1                     2.00          22.92     67.86       985.66   
2                     2.83          26.97     92.22      2201.70   
3                     0.87          13.32     61.57      1357.33   
4                     0.96          20.22     91.11      1538.20   

   Sunlight_Hours  ...  Crop_Type Crop_Growth_Stage  Season Irrigation_Type  \
0            5.90  ...  Sugarcane            Sowing    Zaid            Drip   
1            6.98  ...      Wheat        Vegetative  Kharif         Rainfed   

In [4]:
data.drop('id', axis= 1, inplace= True)

In [5]:
y= data['Irrigation_Need']
X= data.drop('Irrigation_Need', axis= 1)

In [6]:
cat_cols= X.select_dtypes(include= ['object']).columns
num_cols= X.select_dtypes(include= ['float64', 'int64']).columns

print(cat_cols)
print(num_cols)

Index(['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season',
       'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region'],
      dtype='object')
Index(['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity',
       'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours',
       'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm'],
      dtype='object')


In [7]:
X.nunique().sort_values()

Mulching_Used                  2
Season                         3
Crop_Growth_Stage              4
Soil_Type                      4
Water_Source                   4
Irrigation_Type                4
Region                         5
Crop_Type                      6
Organic_Carbon               131
Soil_pH                      341
Electrical_Conductivity      341
Sunlight_Hours               701
Field_Area_hectare          1466
Wind_Speed_kmh              1935
Temperature_C               2934
Soil_Moisture               5223
Humidity                    6475
Previous_Irrigation_mm     10110
Rainfall_mm                19308
dtype: int64

In [8]:
y.value_counts(normalize= True)*100

Irrigation_Need
Low       58.716984
Medium    37.948254
High       3.334762
Name: proportion, dtype: float64

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test= train_test_split(X, y, test_size= 0.2, stratify= y, random_state= 7)

In [10]:
from catboost import CatBoostClassifier

model= CatBoostClassifier(depth= 6, loss_function= 'MultiClass', auto_class_weights= 'Balanced', n_estimators= 500, learning_rate= 0.05, verbose= 100)

cat_cols= list(cat_cols)
model.fit(X_train, y_train, cat_features=cat_cols)


0:	learn: 1.0106140	total: 1.73s	remaining: 14m 21s
100:	learn: 0.1078693	total: 2m 27s	remaining: 9m 44s
200:	learn: 0.0984495	total: 4m 41s	remaining: 6m 59s
300:	learn: 0.0928617	total: 7m 4s	remaining: 4m 40s
400:	learn: 0.0882645	total: 9m 27s	remaining: 2m 20s
499:	learn: 0.0849569	total: 11m 49s	remaining: 0us


CatBoostClassifier(auto_class_weights='Balanced', depth=6, learning_rate=0.05, loss_function='MultiClass', n_estimators=500, verbose=100)

In [11]:
from sklearn.metrics import classification_report, balanced_accuracy_score

preds= model.predict(X_test)
print(classification_report(y_test, preds))
print(balanced_accuracy_score(y_test, preds))

              precision    recall  f1-score   support

        High       0.87      0.95      0.91      4202
         Low       0.99      0.99      0.99     73983
      Medium       0.98      0.97      0.97     47815

    accuracy                           0.98    126000
   macro avg       0.95      0.97      0.96    126000
weighted avg       0.98      0.98      0.98    126000

0.9701611513105927


In [12]:
test_data= pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/test.csv')

In [13]:
test_submission= test_data.drop('id', axis= 1)
test_preds= model.predict(test_submission).ravel()

In [14]:
submission = pd.DataFrame({
    'id': test_data['id'],
    'target': test_preds})

submission.to_csv("submission.csv", index=False)